# Guatemala Dengue Surveillance Analysis, 2012–2024

## Project Overview

### Research Question
How did reported dengue cases change across Guatemala between 2012 and 2024, and what temporal and geographic patterns can be identified using descriptive statistics and statistical inference?

### General Objective
Analyze temporal and geographic patterns of reported dengue cases in Guatemala between 2012 and 2024 using descriptive statistics and statistical methods.

### Specific Objectives
- Identify municipalities with the highest reported number of dengue cases.
- Quantify variability among municipalities.
- Identify municipalities contributing the largest burden.

### Data Sources

| Dataset | Description | Unit of Observation |
|----------|-------------|---------------------|
| municipality | Total reported dengue cases aggregated by municipality for the study period. | One municipality |
| year | Total reported dengue cases in Guatemala by calendar year. | One year |
| year_municipality | Reported dengue cases for each municipality in each year. | One municipality during one year |

### Introduction
This notebook analyzes reported dengue cases in Guatemala from 2012 to 2024. The dataset contains information aggregated by year, municipality, and department. The data were obtained from the Ministerio de Salud Pública y Asistencia Social (MSPAS), Guatemala's national public health authority. The objective of this analysis is to explore temporal trends and the geographic distribution of dengue cases across the country.

## 1. Data Preparation and Quality Assessment

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 

### 1.2 Load Data

In [ ]:
municipality = pd.read_csv('../exports/dengue_municipality.csv')
year_municipality = pd.read_csv('../exports/dengue_year_municipality.csv')
year_totals = pd.read_csv('../exports/dengue_year_totals.csv')


### 1.3 Dataset Dimensions

In [ ]:
print("Municipality dataset:", municipality.shape)
print("Year dataset:", year_totals.shape)
print("Municipality-Year dataset:", year_municipality.shape)

In the municipality dataset, each row represents a municipality and the total number of reported dengue cases observed across the study period. The year dataset contains one row per year, and the municipality-year dataset contains one row for each municipality-year combination available in the source data. The municipality-year dataset contains fewer observations than the theoretical maximum, suggesting that not every municipality has a record for every year.

### 1.4 Data Types

In [ ]:
municipality.info()
year_totals.info()
year_municipality.info()

The variables representing year and reported dengue cases are stored as integers, which is appropriate for numerical analysis. Municipality identifiers are stored as text, reflecting their categorical nature. Based on this review, the data types are suitable for the planned analysis.

### 1.5 Missing Values

In [ ]:
def summarize_missing_values(df, dataset_name):
    summary = pd.DataFrame({
        "dataset": dataset_name,
        "variable": df.columns,
        "missing_values": df.isna().sum().values,
        "missing_percentage": (
            df.isna().mean().values * 100
        ).round(2)
    })

    return summary

In [ ]:
missing_municipality = summarize_missing_values(municipality, "municipality")
missing_year_totals = summarize_missing_values(year_totals, "year_totals")
missing_year_municipality = summarize_missing_values(year_municipality, "year_municipality")

missing_summary = pd.concat(
    [missing_municipality, missing_year_totals, missing_year_municipality],
    ignore_index=True
)

missing_summary 

Missing values were evaluated for every variable in the three datasets to identify incomplete records. No missing values were identified, so no imputation or deletion was required before continuing with exploratory and statistical analysis.

### 1.6 Duplicate Records

In [ ]:
municipality.duplicated(subset=['municipality']).sum()
year_totals.duplicated(subset=['year']).sum()
year_municipality.duplicated(subset=['year', 'municipality']).sum()

No duplicate records were identified in the datasets. Each observation appears to be uniquely represented in the prepared datasets.

## 2. Exploratory Data Analysis

### 2.1 Descriptive Statistics

#### Municipality totals

In [ ]:
municipality["total_cases"].describe()

#### Annual totals

In [ ]:
year_totals["total_cases"].describe()


#### Municipality-year observations

In [ ]:
year_municipality["total_cases"].describe()

#### Interpretation

The descriptive statistics indicate that reported dengue cases are highly dispersed across municipalities and across years. The large difference between the mean and the median, together with a higher standard deviation than the mean, suggests a right-skewed distribution and the presence of a small number of municipalities or years with particularly large case counts.

### 2.2 Distribution Analysis

#### Histogram

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    year_municipality["total_cases"],
    bins=40,
    edgecolor='black',
    alpha=0.8
)

plt.title("Distribution of Reported Dengue Cases per Municipality-Year", fontsize=14)
plt.xlabel("Reported Dengue Cases")
plt.ylabel("Frequency")

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

#### Boxplot

In [ ]:
plt.figure(figsize=(10, 2))

plt.boxplot(year_municipality["total_cases"], vert=False, showmeans=True)

plt.xlim(0, 250)   # Zoom in
plt.title("Distribution of Dengue Cases per Municipality-Year")
plt.xlabel("Reported Cases")

plt.show()

#### Outlier analysis

#### Interpretation

The boxplot reinforces the findings from the histogram and previous descriptive statistics. The distribution of reported dengue cases per municipality-year is strongly right-skewed, with most observations concentrated at low values and a smaller set of observations extending far into the upper tail. These larger values likely reflect outbreak years or municipalities with unusually high dengue burden.

### 2.3 Concentration of Dengue Burden

#### Pareto Analysis

In [ ]:
pareto_df = municipality.sort_values(by="total_cases", ascending=False).reset_index(drop=True)
pareto_df["cumulative_cases"] = pareto_df["total_cases"].cumsum()
total_cases = pareto_df["total_cases"].sum()
pareto_df["cumulative_%"] = (pareto_df["cumulative_cases"]/total_cases) * 100
pareto_df.head(10)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(
    x=range(1, len(pareto_df) + 1),
    height=pareto_df["total_cases"]
)
ax1.set_xlabel("Municipality rank")
ax1.set_ylabel("Total dengue cases")
ax1.set_title(
    "Pareto Analysis of Dengue Cases by Municipality"
)
ax1.set_xlim(1, len(pareto_df))

ax2 = ax1.twinx()
ax2.plot(
    range(1, len(pareto_df) + 1),
    pareto_df["cumulative_%"]
)
ax2.set_ylabel("Cumulative percentage (%)")
ax2.set_ylim(0, 100)

ax2.axhline(
    y=80,
    linestyle="--"
)

threshold_row = pareto_df[
    pareto_df["cumulative_%"] >= 80
].iloc[0]

threshold_rank = threshold_row.name + 1

ax1.axvline(
    x=threshold_rank,
    linestyle="--"
)

#### Interpretation

The Pareto analysis shows that a relatively small subset of municipalities accounts for a large share of the total dengue burden. This concentration suggests that intervention efforts may be most effective when focused on the municipalities that contribute disproportionately to the national case totals.

### 2.4 Temporal Analysis

#### Year-over-Year Changes

In [ ]:
year_totals["case difference"] = year_totals["total_cases"].diff()
year_totals["percent_change"] = year_totals["total_cases"].pct_change() * 100

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))

ax1.plot(
    year_totals["year"],
    year_totals["total_cases"],
    marker='o',
    label="Total Dengue Cases"
)

ax1.set_xlabel("Year")
ax1.set_ylabel("Total Dengue Cases")
ax1.set_title("Annual Dengue Cases and Year-over_Year Change in Guatemala")

ax1.grid(
    axis="y",
    alpha=0.3
)

ax2 = ax1.twinx()
ax2.bar(
    year_totals["year"],
    year_totals["percent_change"],
    alpha=0.25,
    label="Year-over-Year Change (%)"
)

ax2.set_ylabel("Year-over-Year Change (%)")
ax2.axhline(
    y=0,
    linestyle="--",
    color='red'
)  

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper left"
)

important_years = [2019, 2020, 2023]

for year in important_years:
    year_data = year_totals[year_totals["year"] == year].iloc[0]
    x_coordinate, y_coordinate = year_data["year"], year_data["total_cases"]
    
    ax1.annotate(
        text=f"{y_coordinate:,} %",
        xy=(x_coordinate, y_coordinate + 20),
    )

#### Interpretation

The temporal analysis shows that dengue burden was relatively moderate through much of the period, but it rose sharply in 2019 and reached particularly high levels in 2023 and 2024. The year-over-year change chart highlights these turning points and shows how the burden intensified during outbreak periods.

## 3. Municipality-Level Analysis

### 3.1 Top Municipalities by Total Cases

In [ ]:
top_municipalities = municipality.groupby('municipality')['total_cases'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
plt.barh(top_municipalities.index, top_municipalities.values)
plt.xlabel('Total Cases')
plt.ylabel('Municipality')
plt.title('Top 10 Municipalities with Highest Dengue Cases')
plt.xticks(rotation=45)
plt.grid(axis='x', alpha=0.3)
plt.gca().invert_yaxis()  # Invert y-axis to have the highest cases on top
plt.tight_layout()
plt.show()

### 3.2 Persistent High-Burden Municipalities

In [ ]:
year_municipality_renamed = year_municipality.rename(
    columns={"total_cases": "municipality_cases"}
)
year_totals_renamed = year_totals.rename(
    columns={"total_cases": "national_cases"}
)
municipality_national_cases = pd.merge(year_municipality_renamed, year_totals_renamed, on="year")
municipality_national_cases['municipality_share'] = municipality_national_cases['municipality_cases'] / municipality_national_cases['national_cases']
persistent_driven_burden = municipality_national_cases.groupby("municipality")["municipality_share"].agg(
    average_share="mean",
    maximum_share="max"
).reset_index()
municipality_top10_persistent_burden = persistent_driven_burden.sort_values("average_share", ascending=False).head(10)
municipality_top10_persistent_burden.plot(
    x="municipality",
    y=["average_share", "maximum_share"],
    kind="barh",
    figsize=(10, 6)
)
plt.title("Persistent and Peak Dengue Burden by Municipality (2012–2024)")
plt.xlabel("Share of National Cases")
plt.ylabel("Municipality")
plt.legend(title="Share Type")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3.3 Municipality Rankings Through Time

The municipality-level burden analysis indicates that several municipalities consistently contribute to a meaningful share of national dengue cases, while some municipalities become particularly prominent during outbreak years.

### 3.4 Interpretation

The maximum share is higher than the average share for all top municipalities, suggesting that dengue burden fluctuates substantially over time. However, municipalities such as Coatepeque and Guatemala also maintain relatively high average shares, indicating persistent contribution in addition to outbreak-year peaks.

## 4. Conclusions

1. Dengue cases stayed relatively low throughout 2012 to 2022, with a major spike in 2019 and a large outbreak period in 2023 and 2024.
2. Reported dengue cases are unevenly distributed across municipalities, with a relatively small subset of municipalities accounting for a disproportionately large share of the total burden.
3. Several municipalities contribute consistently to the national dengue burden, rather than the burden being concentrated in a single municipality.
4. The temporal pattern suggests that outbreaks are episodic but concentrated in a limited number of municipalities and years.

## 5. Limitations and Future Work

This notebook uses aggregate municipality and annual data and therefore cannot distinguish between changes in true transmission and changes in reporting behavior. Future work could extend the analysis with department-level comparisons, time-series modeling, and more detailed epidemiological context to better explain the observed peaks and geographic concentration of dengue burden.